# 06.5 Dynamic topic modeling con BERTopic (Híbrido)

Este cuaderno prepara un flujo para modelar temas dinámicos sobre los posteos de Revista Anfibia usando una arquitectura híbrida:
- **Embeddings/Clustering:** Usa el texto original para capturar la sintaxis y semántica.
- **Representación (Palabras clave):** Usa el texto lematizado para evitar redundancia morfológica en español.

In [20]:
# EJECUTAR ESTA CELDA PRIMERO QUE NADA (Y REINICIAR EL KERNEL ANTES SI FALLÓ)
import coverage.types as ct

# Parche de compatibilidad para evitar que Numba/UMAP crasheen
for name, fallback in {
    'Tracer': getattr(ct, 'TTracer', object),
    'TShouldTraceFn': object,
    'TShouldStartContextFn': object,
    'TFileDisposition': object,
}.items():
    if not hasattr(ct, name):
        setattr(ct, name, fallback)

## 1. Configuración del entorno

In [21]:
import os
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from umap import UMAP
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired
from bertopic import BERTopic

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "corpus_anfibia_lematizado.csv"
STOPWORDS_PATH = PROJECT_ROOT / "data" / "processed" / "stopwords_anfibia.txt"

assert DATA_PATH.exists(), f"Falta archivo de entrada: {DATA_PATH.name}"
assert STOPWORDS_PATH.exists(), f"Falta archivo de stopwords: {STOPWORDS_PATH.name}"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "bertopic_hibrido"
GRAFICOS_DIR = OUTPUT_DIR / "graficos"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
GRAFICOS_DIR.mkdir(exist_ok=True, parents=True)

## 2. Carga de Corpus y Stopwords
Preparamos los textos, normalizamos categorías y cargamos la lista de stopwords.

In [22]:
MAX_DOCUMENTS = 4000 ## el corpus igual tiene ~3600
USE_COLUMNS = [
    "doc_id", 
    "ID", "source_url", "title_text", "tagline", "autor_names",
    "post_date", "category", "post_tag", "text_type",
    "document_text_original", "document_text_final"
]

df = pd.read_csv(DATA_PATH, usecols=USE_COLUMNS, nrows=MAX_DOCUMENTS)

print(f"Documentos listos: {df.shape[0]}")

# Normalizar categorías vectorizado
def split_categories(val):
    return [p.strip() for p in re.split(r"[;,]", str(val)) if p.strip()]

categories_df = (
    df[["doc_id", "category"]]
    .assign(category_normalized=df["category"].map(split_categories))
    .explode("category_normalized")
    .fillna({"category_normalized": "Sin categoria"})
    [["doc_id", "category_normalized"]]
)

# Textos, fechas y stopwords
docs_naturales = df["document_text_original"].tolist()
docs_lematizados = df["document_text_final"].tolist()
timestamps = pd.to_datetime(df["post_date"].str.split(' ').str[0], format="%Y-%m-%d", errors="coerce")

with open(STOPWORDS_PATH, "r", encoding="utf-8") as f:
    STOPWORDS_LIST = f.read().splitlines()

Documentos listos: 3556


## 3. Entrenamiento Híbrido y Reducción de Outliers (Instancia Única)

1. Calculamos embeddings sobre el texto.
2. Clusterizamos (UMAP + HDBSCAN).
3. Reasignamos las notas outliers (-1).
4. Extraemos las palabras clave con el texto lematizado y las stopwords.

In [23]:
print("1. Calculando embeddings originales...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings_naturales = embedding_model.encode(docs_naturales, show_progress_bar=True)

print("\n2. Entrenando modelo base (HDBSCAN puro)...")
vectorizer_model = CountVectorizer(stop_words=STOPWORDS_LIST, min_df=3)
representation_model = KeyBERTInspired()
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

topic_model = BERTopic(
    umap_model=umap_model,
    embedding_model=embedding_model,
    language="spanish",
    calculate_probabilities=True,
    min_topic_size=10,
    nr_topics=None,
    representation_model=representation_model,
    vectorizer_model=vectorizer_model,
)

topics_base, probs = topic_model.fit_transform(docs_lematizados, embeddings=embeddings_naturales)

print("\n3. Reasignando notas outliers (-1)...")
topics_reducidos = topic_model.reduce_outliers(
    docs_lematizados,
    topics_base,
    strategy="embeddings",
    embeddings=embeddings_naturales
)

print("\n4. Recalibrando palabras clave lematizadas de los tópicos (c-TF-IDF)...")
# Hay que pasarle el vectorizador de nuevo sí o sí
topic_model.update_topics(
    docs_lematizados, 
    topics=topics_reducidos,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model
)

topic_model.save(OUTPUT_DIR / "bertopic_hibrido_model") 
topics = topics_reducidos
print("\nModelo optimizado y guardado exitosamente.")

1. Calculando embeddings originales...


Batches: 100%|██████████| 112/112 [01:35<00:00,  1.18it/s]



2. Entrenando modelo base (HDBSCAN puro)...


2026-10-02 11:59:14,327 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.



3. Reasignando notas outliers (-1)...

4. Recalibrando palabras clave lematizadas de los tópicos (c-TF-IDF)...


2026-10-02 11:59:31,875 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.



Modelo optimizado y guardado exitosamente.


## 4. Diagnóstico visual de Outliers
Comparamos cuánto ruido generó el modelo base y cómo quedó tras la reasignación.

In [24]:
outliers_base = topics_base.count(-1)
clasificados_base = len(topics_base) - outliers_base
outliers_red = topics_reducidos.count(-1)
clasificados_red = len(topics_reducidos) - outliers_red

fig = make_subplots(
    rows=1, cols=2, 
    specs=[[{'type': 'domain'}, {'type': 'domain'}]],
    subplot_titles=[
        f"Puro",
        f"Post-Reducción"
    ]
)

fig.add_trace(go.Pie(labels=["Ruido (-1)", "Válidos"], values=[outliers_base, clasificados_base], hole=0.4, marker=dict(colors=["#d62728", "#2ca02c"])), row=1, col=1)
fig.add_trace(go.Pie(labels=["Ruido (-1)", "Válidos"], values=[outliers_red, clasificados_red], hole=0.4, marker=dict(colors=["#d62728", "#2ca02c"])), row=1, col=2)

fig.update_layout(title_text="<b>Comparacion de topicos</b>", template="plotly_white", height=450, width=800)
fig.show()

## 5. Extracción de Evolución Temporal y Gráficos Básicos

In [ ]:
print("Calculando evolución temporal de los tópicos...")
topics_over_time = topic_model.topics_over_time(
    docs=docs_lematizados,
    timestamps=timestamps.dt.to_pydatetime().tolist(),
    #nr_bins=20,
    nr_bins=40, #cada uno abarca entre 3-4 meses
)
topics_over_time.to_csv(OUTPUT_DIR / "topics_over_time.csv", index=False)

# 1. Asignar al df original 
df['topic'] = topics

fig = topic_model.visualize_barchart(top_n_topics=24, n_words=8, title="<b>Palabras representativas (Híbrido)</b>")
colors = px.colors.qualitative.G10
for i, trace in enumerate(fig.data):
    trace.marker.color = colors[i % len(colors)]

fig.update_traces(hovertemplate="<b>Término:</b> %{y}<br><b>Relevancia:</b> %{x:.3f}<extra></extra>")
fig.for_each_xaxis(lambda ax: ax.update(title_text="", showticklabels=False))
fig.for_each_annotation(lambda a: a.update(text=f"<b>{a.text.split('_')[0]}</b>"))
fig.update_layout(height=950, width=1300, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='#ffffff')

fig.write_html(GRAFICOS_DIR / "01_barchart_topicos.html")
fig.show()

Calculando evolución temporal de los tópicos...


In [26]:
# 1. Generar la estructura de agrupamiento jerárquico
hierarchical_topics = topic_model.hierarchical_topics(docs_lematizados)

100%|██████████| 42/42 [00:27<00:00,  1.52it/s]


In [ ]:
# 2. Visualizar la jerarquia
fig_tree = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig_tree.show()
fig_tree.write_html("topicos_agrupacion.html")

In [31]:
tree = topic_model.get_topic_tree(hierarchical_topics)
print(tree)

.
├─feminista_feminismo_escritor_político_periodista
│    ├─periodista_escritor_literario_autor_ficción
│    │    ├─■──padre_papá_hijo_abuelo_hermano ── Topic: 42
│    │    └─literario_periodista_autor_escritor_literatura
│    │         ├─periodista_escritor_literario_autor_literatura
│    │         │    ├─■──fifa_futbolista_fútbol_deporte_brasil ── Topic: 2
│    │         │    └─literario_poeta_escritor_autor_literatura
│    │         │         ├─■──periodista_periodístico_periódico_narrativo_escritora ── Topic: 19
│    │         │         └─poeta_literario_autor_escritor_poesía
│    │         │              ├─literario_poeta_escritora_libro_escritor
│    │         │              │    ├─■──ficción_protagonista_personaje_melodrama_película ── Topic: 29
│    │         │              │    └─literario_escritora_autor_literatura_escritor
│    │         │              │         ├─■──película_personaje_actor_hombre_américo ── Topic: 34
│    │         │              │         └─■──literario_e

In [ ]:
from scipy.cluster import hierarchy
import copy
topic_model_prueba = copy.deepcopy(topic_model)


# Average con distancia Coseno / Defecto
hierarchical_topics_avg = topic_model_prueba.hierarchical_topics(
    docs_lematizados,
    linkage_function=lambda x: hierarchy.linkage(x, method='average', metric='cosine')
)

# # Solo distancia coseno
# hierarchical_topics_comp = topic_model.hierarchical_topics(
#     docs_lematizados,
#     linkage_function=lambda x: hierarchy.linkage(x, method='complete', metric='cosine')
# )

fig = topic_model_prueba.visualize_hierarchy(hierarchical_topics=hierarchical_topics_avg)
fig.show()

100%|██████████| 42/42 [00:28<00:00,  1.46it/s]


In [36]:
tree = topic_model_prueba.get_topic_tree(hierarchical_topics)
print(tree)

.
├─feminista_feminismo_escritor_político_periodista
│    ├─periodista_escritor_literario_autor_ficción
│    │    ├─■──padre_papá_hijo_abuelo_hermano ── Topic: 42
│    │    └─literario_periodista_autor_escritor_literatura
│    │         ├─periodista_escritor_literario_autor_literatura
│    │         │    ├─■──fifa_futbolista_fútbol_deporte_brasil ── Topic: 2
│    │         │    └─literario_poeta_escritor_autor_literatura
│    │         │         ├─■──periodista_periodístico_periódico_narrativo_escritora ── Topic: 19
│    │         │         └─poeta_literario_autor_escritor_poesía
│    │         │              ├─literario_poeta_escritora_libro_escritor
│    │         │              │    ├─■──ficción_protagonista_personaje_melodrama_película ── Topic: 29
│    │         │              │    └─literario_escritora_autor_literatura_escritor
│    │         │              │         ├─■──película_personaje_actor_hombre_américo ── Topic: 34
│    │         │              │         └─■──literario_e

In [41]:
tree = topic_model_prueba.get_topic_tree(hierarchical_topics)
print(tree)

.
├─feminista_feminismo_escritor_político_periodista
│    ├─periodista_escritor_literario_autor_ficción
│    │    ├─■──padre_papá_hijo_abuelo_hermano ── Topic: 42
│    │    └─literario_periodista_autor_escritor_literatura
│    │         ├─periodista_escritor_literario_autor_literatura
│    │         │    ├─■──fifa_futbolista_fútbol_deporte_brasil ── Topic: 2
│    │         │    └─literario_poeta_escritor_autor_literatura
│    │         │         ├─■──periodista_periodístico_periódico_narrativo_escritora ── Topic: 19
│    │         │         └─poeta_literario_autor_escritor_poesía
│    │         │              ├─literario_poeta_escritora_libro_escritor
│    │         │              │    ├─■──ficción_protagonista_personaje_melodrama_película ── Topic: 29
│    │         │              │    └─literario_escritora_autor_literatura_escritor
│    │         │              │         ├─■──película_personaje_actor_hombre_américo ── Topic: 34
│    │         │              │         └─■──literario_e

In [ ]:
fig_topics = topic_model.visualize_topics(  )
fig_topics.show()

In [50]:
topic_model.visualize_barchart()

In [51]:
topic_model.visualize_heatmap()